# Step 10 — Final inclusive T-cell and Treg rescue

This notebook is the final sensitivity-oriented pass. It does **not** rerun
ResolVI, pyUCell, integration, Harmony, or Leiden.

It starts from the completed Step 08 fallback objects and preserves the existing
strict calls. It adds a deliberately inclusive but auditable rescue hierarchy.

## Why another rescue tier is justified

The strict Step 08 hierarchy requires a T-cell score to be sample-high **and**
to exceed every competing non-T lineage program. That is appropriate for a
high-specificity tier, but it can reject real T cells in spatially mixed
profiles.

The independent 2-µm-bin survey and the segmented raw-count audit show
substantial FOXP3 and CD8A signal in samples where the strict hierarchy called
zero T cells. This notebook therefore separates:

```text
strict T lineage
primary rescued T lineage
exploratory rescued T lineage
```

rather than replacing one hard threshold with another.

## T-lineage rescue

### Strict tier

The existing Step 08:

```python
fallback_T_lineage
```

is retained unchanged.

### Primary rescue tier

A new T cell must have:

```text
sample-high corrected T-core score
+
coherent raw T evidence or strong two-reference T support
+
T percentile reasonably close to the strongest competing lineage percentile
```

Coherent raw T evidence requires:

```text
at least one TCR-domain gene:
    TRAC, TRBC1, TRBC2

AND

at least one CD3-domain gene:
    CD3D, CD3E, CD247

OR at least three T-core genes total
```

### Exploratory rescue tier

This tier is intentionally more sensitive. It requires a lower T-core score
threshold plus either coherent raw T evidence or reference-supported T
evidence. It does not require the T score to beat the mixed non-T programs.

Cells remain labeled as clean or spatially mixed in a separate column.

## Treg rescue

The old impossible zero-MAD cutoff is not reused. Thresholds are percentile
based with an absolute floor.

Three Treg tiers are saved:

```text
Treg_high_confidence
    primary T cell
    + high Treg-core score
    + CD4-compatible
    + not CD8-dominant
    + raw FOXP3 or IL2RA+CTLA4 support

Treg_supported
    primary T cell
    + high Treg-core score
    + CD4-compatible
    + not CD8-dominant
    raw support not required

Treg_exploratory
    exploratory T cell
    + mildly high Treg-core score
    + CD4-compatible
    + not strongly CD8-dominant
    + either raw FOXP3/raw anchor or the stricter Treg score threshold
```

The primary result uses `Treg_supported`. `Treg_exploratory` is a sensitivity
analysis and should be reported as such.

## 2-µm-bin reference

The supplied FOXP3/CD8A bin counts are embedded as an independent audit table.
They are **not** used to choose thresholds or force target counts.

## Main outputs

For every sample:

```text
full annotated H5AD
primary rescued-T H5AD
exploratory rescued-T H5AD
cell-level metadata Parquet
spatial metadata Parquet
strict/primary/exploratory count table
threshold table
candidate-cell ID tables
spatial and score diagnostic figures
```

Cross-sample outputs compare:

```text
strict versus rescued T counts
strict versus supported/exploratory Treg counts
raw FOXP3-positive cells
FOXP3-positive 2-µm bins
paired Screen versus C2D15 directions
```

In [1]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import json
import math
import re
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

print("anndata:", ad.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

anndata: 0.12.19
numpy: 1.26.4
pandas: 2.3.3


In [2]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057"
)
PIPELINE_ROOT = (
    PROJECT_ROOT
    / "tmp"
    / "proseg_resolvi_immune_enrichment_v1"
)

FALLBACK_ROOT = (
    PIPELINE_ROOT
    / "08_fallback_ucell_hierarchical_annotation"
)
OUTPUT_ROOT = (
    PIPELINE_ROOT
    / "10_final_inclusive_T_Treg_rescue"
)
OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SAMPLE_INFO = {
    "Screen_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "Screen",
    },
    "C2D15_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "C2D15",
    },
}
SECTION_NAMES = list(
    SAMPLE_INFO
)

# Independent pre-segmentation reference. These values are used only in
# after-the-fact audit tables and figures.
BIN_REFERENCE = {
    "Screen_18_23": {
        "FOXP3_positive_bins": 33,
        "FOXP3_cells_bins_div3": 11,
        "CD8A_positive_bins": 80,
        "CD8A_cells_bins_div3": 26,
    },
    "C2D15_18_23": {
        "FOXP3_positive_bins": 79,
        "FOXP3_cells_bins_div3": 26,
        "CD8A_positive_bins": 327,
        "CD8A_cells_bins_div3": 109,
    },
    "Screen_23_25": {
        "FOXP3_positive_bins": 22,
        "FOXP3_cells_bins_div3": 7,
        "CD8A_positive_bins": 170,
        "CD8A_cells_bins_div3": 56,
    },
    "C2D15_23_25": {
        "FOXP3_positive_bins": 5,
        "FOXP3_cells_bins_div3": 1,
        "CD8A_positive_bins": 187,
        "CD8A_cells_bins_div3": 62,
    },
    "Screen_39_21": {
        "FOXP3_positive_bins": 191,
        "FOXP3_cells_bins_div3": 64,
        "CD8A_positive_bins": 444,
        "CD8A_cells_bins_div3": 148,
    },
    "C2D15_39_21": {
        "FOXP3_positive_bins": 111,
        "FOXP3_cells_bins_div3": 37,
        "CD8A_positive_bins": 103,
        "CD8A_cells_bins_div3": 34,
    },
    "Screen_16_22": {
        "FOXP3_positive_bins": 80,
        "FOXP3_cells_bins_div3": 26,
        "CD8A_positive_bins": 267,
        "CD8A_cells_bins_div3": 89,
    },
    "C2D15_16_22": {
        "FOXP3_positive_bins": 327,
        "FOXP3_cells_bins_div3": 109,
        "CD8A_positive_bins": 623,
        "CD8A_cells_bins_div3": 207,
    },
    "Screen_17_26": {
        "FOXP3_positive_bins": 170,
        "FOXP3_cells_bins_div3": 56,
        "CD8A_positive_bins": 303,
        "CD8A_cells_bins_div3": 101,
    },
    "C2D15_17_26": {
        "FOXP3_positive_bins": 187,
        "FOXP3_cells_bins_div3": 62,
        "CD8A_positive_bins": 337,
        "CD8A_cells_bins_div3": 112,
    },
    "Screen_30_16": {
        "FOXP3_positive_bins": 444,
        "FOXP3_cells_bins_div3": 148,
        "CD8A_positive_bins": 1615,
        "CD8A_cells_bins_div3": 538,
    },
    "C2D15_30_16": {
        "FOXP3_positive_bins": 103,
        "FOXP3_cells_bins_div3": 34,
        "CD8A_positive_bins": 393,
        "CD8A_cells_bins_div3": 131,
    },
}

# T-cell rescue score thresholds. These are quantile/floor rules; the previous
# zero-MAD robust cutoff is deliberately not used.
PRIMARY_T_CORE_QUANTILE = 0.85
EXPLORATORY_T_CORE_QUANTILE = 0.75
T_CORE_ABSOLUTE_FLOOR = 0.03

# Percentile competition is used only for the primary tier.
PRIMARY_T_VS_NON_T_PERCENTILE_ALLOWANCE = 0.10
PRIMARY_REFERENCE_T_ALLOWANCE = 0.15

# Treg tiers.
SUPPORTED_TREG_QUANTILE = 0.95
EXPLORATORY_TREG_QUANTILE = 0.90
TREG_ABSOLUTE_FLOOR = 0.03

# Percentile margins are on a 0–1 within-sample scale.
SUPPORTED_TREG_VS_CD8_ALLOWANCE = 0.10
EXPLORATORY_TREG_VS_CD8_ALLOWANCE = 0.20
CD4_COMPATIBILITY_PERCENTILE = 0.50

# CD4/CD8 subtype labeling within rescued T cells.
CD4_CD8_POSITIVE_PERCENTILE = 0.60
CD4_CD8_DOMINANCE_MARGIN = 0.10
CD8_VS_NK_ALLOWANCE = 0.15

# The primary T-only H5AD includes strict + primary rescue.
# The exploratory T-only H5AD includes all exploratory rescues as well.
WRITE_FULL_ANNOTATED_H5AD = True
WRITE_PRIMARY_T_H5AD = True
WRITE_EXPLORATORY_T_H5AD = True
WRITE_METADATA_PARQUET = True
WRITE_SPATIAL_PARQUET = True
H5AD_COMPRESSION = "lzf"

PLOT_DPI = 500
PLOT_MAX_CELLS = 200_000
RANDOM_STATE = 0

CONTINUE_ON_ERROR = True
PIPELINE_VERSION = (
    "2026-08-02-final-inclusive-T-Treg-rescue-v1"
)

print("Samples:", SECTION_NAMES)
print("Output root:", OUTPUT_ROOT)

Samples: ['Screen_39_21', 'C2D15_39_21', 'Screen_17_26', 'C2D15_17_26', 'Screen_18_23', 'C2D15_18_23', 'Screen_16_22', 'C2D15_16_22', 'Screen_30_16', 'C2D15_30_16', 'Screen_23_25', 'C2D15_23_25']
Output root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue


In [3]:
# ---------------------------------------------------------------------
# Marker domains
# ---------------------------------------------------------------------
TCR_DOMAIN = [
    "TRAC",
    "TRBC1",
    "TRBC2",
]
CD3_DOMAIN = [
    "CD3D",
    "CD3E",
    "CD247",
]
T_SUPPORTING = [
    "CD2",
    "CD7",
    "LCK",
    "LAT",
    "IL32",
    "LTB",
]
T_CORE_ALL = list(
    dict.fromkeys(
        TCR_DOMAIN
        + CD3_DOMAIN
        + T_SUPPORTING
    )
)

RAW_REFERENCE_GENES = list(
    dict.fromkeys(
        T_CORE_ALL
        + [
            "FOXP3",
            "IL2RA",
            "CTLA4",
            "CD4",
            "CD8A",
        ]
    )
)

BROAD_SCORE_COLUMNS = {
    "T_core": "fallback_T_core_score",
    "Tumor": "fallback_Tumor_score",
    "Endothelial": "fallback_Endothelial_score",
    "Monocyte_macrophage": (
        "fallback_Monocyte_macrophage_score"
    ),
    "Fibroblast": "fallback_Fibroblast_score",
    "B_plasma": "fallback_B_plasma_score",
    "NK": "fallback_NK_score",
}
NON_T_BROAD_KEYS = [
    "Tumor",
    "Endothelial",
    "Monocyte_macrophage",
    "Fibroblast",
    "B_plasma",
    "NK",
]

SUBTYPE_SCORE_COLUMNS = {
    "CD4": "fallback_CD4_subtype_score",
    "CD8": "fallback_CD8_subtype_score",
    "Treg": "fallback_Treg_subtype_score",
    "NK": "fallback_NK_score",
}

In [4]:
# ---------------------------------------------------------------------
# Paths and portable H5AD I/O
# ---------------------------------------------------------------------
def paths_for_sample(
    sample: str,
) -> dict[str, Path]:
    source_root = (
        FALLBACK_ROOT
        / sample
    )
    out = (
        OUTPUT_ROOT
        / sample
    )
    figures = out / "figures"

    out.mkdir(
        parents=True,
        exist_ok=True,
    )
    figures.mkdir(
        parents=True,
        exist_ok=True,
    )

    return {
        "source_root": source_root,
        "source_h5ad": (
            source_root
            / f"{sample}_fallback_ucell_annotated.h5ad"
        ),
        "source_metadata": (
            source_root
            / f"{sample}_fallback_ucell_metadata.parquet"
        ),
        "out": out,
        "figures": figures,
        "annotated_h5ad": (
            out
            / f"{sample}_inclusive_rescue_annotated.h5ad"
        ),
        "primary_T_h5ad": (
            out
            / f"{sample}_primary_rescued_Tcells.h5ad"
        ),
        "exploratory_T_h5ad": (
            out
            / f"{sample}_exploratory_rescued_Tcells.h5ad"
        ),
        "metadata": (
            out
            / f"{sample}_inclusive_rescue_metadata.parquet"
        ),
        "spatial": (
            out
            / f"{sample}_inclusive_rescue_spatial.parquet"
        ),
        "counts": (
            out
            / f"{sample}_inclusive_rescue_counts.csv"
        ),
        "thresholds": (
            out
            / f"{sample}_inclusive_rescue_thresholds.csv"
        ),
        "candidate_ids": (
            out
            / f"{sample}_inclusive_rescue_candidate_ids.csv"
        ),
        "summary": (
            out
            / f"{sample}_inclusive_rescue_summary.json"
        ),
    }


def write_json(
    payload,
    path: str | Path,
) -> None:
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    temporary.replace(path)


def _is_nullable_string_series(
    series: pd.Series,
) -> bool:
    return (
        isinstance(
            series.dtype,
            pd.StringDtype,
        )
        or type(
            series.array
        ).__name__
        in {
            "StringArray",
            "ArrowStringArray",
        }
        or str(
            series.dtype
        ) == "str"
        or str(
            series.dtype
        ).startswith(
            "string"
        )
    )


def _legacy_string_frame(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    output = frame.copy()

    for column in output.columns:
        series = output[column]

        if _is_nullable_string_series(
            series
        ):
            output[column] = (
                series
                .fillna("")
                .astype(str)
                .astype(object)
            )
            continue

        if pd.api.types.is_object_dtype(
            series.dtype
        ):
            def _safe_text(value):
                if (
                    value is None
                    or value is pd.NA
                ):
                    return ""
                if isinstance(
                    value,
                    bytes,
                ):
                    return value.decode(
                        "utf-8",
                        errors="replace",
                    )
                if isinstance(
                    value,
                    str,
                ):
                    return value
                if isinstance(
                    value,
                    np.generic,
                ):
                    value = value.item()
                if isinstance(
                    value,
                    (
                        dict,
                        list,
                        tuple,
                        set,
                        np.ndarray,
                        Path,
                    ),
                ):
                    return json.dumps(
                        value,
                        default=str,
                        sort_keys=True,
                    )
                return str(value)

            output[column] = (
                series
                .map(
                    _safe_text
                )
                .astype(object)
            )

    if (
        str(
            output.index.dtype
        ) == "str"
        or str(
            output.index.dtype
        ).startswith(
            "string"
        )
        or type(
            output.index.array
        ).__name__
        in {
            "StringArray",
            "ArrowStringArray",
        }
    ):
        name = output.index.name
        output.index = pd.Index(
            pd.Series(
                output.index,
                dtype="string",
            )
            .fillna("")
            .astype(str)
            .to_numpy(
                dtype=object
            ),
            name=name,
        )

    return output


def safe_write_h5ad(
    adata_object: ad.AnnData,
    filename: str | Path,
    *,
    compression: str = "lzf",
) -> None:
    filename = Path(filename)
    filename.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    adata_object.obs = (
        _legacy_string_frame(
            adata_object.obs
        )
    )
    adata_object.var = (
        _legacy_string_frame(
            adata_object.var
        )
    )

    temporary = filename.with_name(
        f"{filename.stem}.tmp{filename.suffix}"
    )
    if temporary.exists():
        temporary.unlink()

    with ad.settings.override(
        allow_write_nullable_strings=False
    ):
        adata_object.write_h5ad(
            temporary,
            compression=compression,
            convert_strings_to_categoricals=False,
        )

    temporary.replace(
        filename
    )
    print("Saved:", filename)


def choose_spatial_key(
    adata: ad.AnnData,
) -> str | None:
    for key in (
        "X_spatial",
        "spatial",
        "spatial_fullres",
    ):
        if key in adata.obsm:
            value = np.asarray(
                adata.obsm[key]
            )
            if (
                value.ndim == 2
                and value.shape[0]
                == adata.n_obs
                and value.shape[1]
                >= 2
            ):
                return key
    return None

In [5]:
# ---------------------------------------------------------------------
# Raw-count and reference-label evidence
# ---------------------------------------------------------------------
def map_present(
    var_names: pd.Index,
    genes: list[str],
) -> dict[str, str | None]:
    lookup = {
        str(name).upper(): str(name)
        for name in var_names
    }
    return {
        gene: lookup.get(
            gene.upper()
        )
        for gene in genes
    }


def add_raw_evidence(
    adata: ad.AnnData,
) -> dict[str, str | None]:
    mapping = map_present(
        adata.var_names,
        RAW_REFERENCE_GENES,
    )

    for gene, mapped in mapping.items():
        count_column = (
            f"rescue_raw_{gene}_count"
        )
        detected_column = (
            f"rescue_raw_{gene}_detected"
        )

        if mapped is None:
            values = np.zeros(
                adata.n_obs,
                dtype=np.int32,
            )
        else:
            matrix = adata[
                :,
                [mapped],
            ].X
            if sp.issparse(
                matrix
            ):
                values = np.asarray(
                    matrix.sum(
                        axis=1
                    )
                ).ravel()
            else:
                values = np.asarray(
                    matrix
                ).ravel()

            values = np.rint(
                values
            ).astype(
                np.int32
            )

        adata.obs[
            count_column
        ] = values
        adata.obs[
            detected_column
        ] = (
            values > 0
        )

    tcr_count = np.column_stack(
        [
            adata.obs[
                f"rescue_raw_{gene}_detected"
            ].to_numpy(
                dtype=bool
            )
            for gene in TCR_DOMAIN
        ]
    ).sum(axis=1)
    cd3_count = np.column_stack(
        [
            adata.obs[
                f"rescue_raw_{gene}_detected"
            ].to_numpy(
                dtype=bool
            )
            for gene in CD3_DOMAIN
        ]
    ).sum(axis=1)
    T_core_count = np.column_stack(
        [
            adata.obs[
                f"rescue_raw_{gene}_detected"
            ].to_numpy(
                dtype=bool
            )
            for gene in T_CORE_ALL
        ]
    ).sum(axis=1)

    adata.obs[
        "rescue_raw_TCR_domain_count"
    ] = tcr_count.astype(
        np.int16
    )
    adata.obs[
        "rescue_raw_CD3_domain_count"
    ] = cd3_count.astype(
        np.int16
    )
    adata.obs[
        "rescue_raw_T_core_gene_count"
    ] = T_core_count.astype(
        np.int16
    )
    adata.obs[
        "rescue_raw_T_coherent"
    ] = (
        (
            tcr_count >= 1
        )
        & (
            cd3_count >= 1
        )
    ) | (
        T_core_count >= 3
    )
    adata.obs[
        "rescue_raw_T_sensitive"
    ] = (
        T_core_count >= 2
    )

    adata.obs[
        "rescue_raw_Treg_anchor"
    ] = (
        adata.obs[
            "rescue_raw_FOXP3_detected"
        ].to_numpy(
            dtype=bool
        )
        | (
            adata.obs[
                "rescue_raw_IL2RA_detected"
            ].to_numpy(
                dtype=bool
            )
            & adata.obs[
                "rescue_raw_CTLA4_detected"
            ].to_numpy(
                dtype=bool
            )
        )
    )

    return mapping


def combine_text_columns(
    obs: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    columns = [
        column
        for column in columns
        if column in obs.columns
    ]
    if not columns:
        return pd.Series(
            "",
            index=obs.index,
            dtype="string",
        )

    output = (
        obs[
            columns[0]
        ]
        .astype("string")
        .fillna("")
    )
    for column in columns[1:]:
        output = (
            output
            + " | "
            + obs[
                column
            ]
            .astype("string")
            .fillna("")
        )

    return output.str.lower()


def text_supports_T(
    text: pd.Series,
) -> np.ndarray:
    return (
        text.str.contains(
            (
                r"\b(t cells?|t-cell|cd4|cd8|treg|"
                r"regulatory t|helper t|cytotoxic t|"
                r"tcm|tem|mait|gamma.?delta)\b"
            ),
            regex=True,
            na=False,
        )
        .to_numpy(
            dtype=bool
        )
    )


def add_reference_T_evidence(
    adata: ad.AnnData,
) -> dict:
    panhuman_text = combine_text_columns(
        adata.obs,
        [
            "full_hierarchical_labels",
            "final_level_labels",
            "azimuth_broad",
            "azimuth_medium",
            "azimuth_fine",
        ],
    )
    celltypist_text = combine_text_columns(
        adata.obs,
        [
            "celltypist_high_top_label",
            "celltypist_high_prob_match_label",
            "celltypist_low_top_label",
            "celltypist_low_prob_match_label",
        ],
    )

    panhuman_T = text_supports_T(
        panhuman_text
    )
    celltypist_T = text_supports_T(
        celltypist_text
    )

    boolean_candidates = [
        column
        for column in (
            "panhuman_t_candidate",
            "celltypist_high_t_candidate",
            "celltypist_low_t_candidate",
            "corrected_t_score_candidate",
            "t_candidate_multi_evidence",
            "T_candidate_multi_evidence",
        )
        if column in adata.obs.columns
    ]

    extra_support = np.zeros(
        adata.n_obs,
        dtype=bool,
    )
    for column in boolean_candidates:
        extra_support |= (
            adata.obs[
                column
            ]
            .fillna(False)
            .to_numpy(
                dtype=bool
            )
        )

    adata.obs[
        "rescue_reference_panhuman_T"
    ] = panhuman_T
    adata.obs[
        "rescue_reference_celltypist_T"
    ] = celltypist_T
    adata.obs[
        "rescue_reference_both_T"
    ] = (
        panhuman_T
        & celltypist_T
    )
    adata.obs[
        "rescue_reference_any_T"
    ] = (
        panhuman_T
        | celltypist_T
        | extra_support
    )

    return {
        "boolean_reference_columns": (
            boolean_candidates
        ),
        "n_panhuman_T": int(
            panhuman_T.sum()
        ),
        "n_celltypist_T": int(
            celltypist_T.sum()
        ),
        "n_both_T": int(
            (
                panhuman_T
                & celltypist_T
            ).sum()
        ),
    }

In [6]:
# ---------------------------------------------------------------------
# Percentile and threshold helpers
# ---------------------------------------------------------------------
def percentile_rank(
    values: np.ndarray,
    mask: np.ndarray | None = None,
) -> np.ndarray:
    values = np.asarray(
        values,
        dtype=float,
    )

    if mask is None:
        mask = np.ones(
            len(values),
            dtype=bool,
        )
    else:
        mask = np.asarray(
            mask,
            dtype=bool,
        )

    finite = (
        mask
        & np.isfinite(
            values
        )
    )
    output = np.full(
        len(values),
        np.nan,
        dtype=np.float32,
    )

    if finite.sum() == 0:
        return output

    output[
        finite
    ] = (
        pd.Series(
            values[
                finite
            ]
        )
        .rank(
            method="average",
            pct=True,
        )
        .to_numpy(
            dtype=np.float32
        )
    )

    return output


def quantile_floor_threshold(
    values: np.ndarray,
    *,
    quantile: float,
    floor: float,
    mask: np.ndarray | None = None,
) -> dict:
    values = np.asarray(
        values,
        dtype=float,
    )

    if mask is None:
        mask = np.ones(
            len(values),
            dtype=bool,
        )
    else:
        mask = np.asarray(
            mask,
            dtype=bool,
        )

    finite = (
        mask
        & np.isfinite(
            values
        )
    )

    if finite.sum() == 0:
        threshold = float(
            "inf"
        )
        quantile_cut = float(
            "nan"
        )
    else:
        quantile_cut = float(
            np.quantile(
                values[
                    finite
                ],
                float(
                    quantile
                ),
            )
        )
        threshold = max(
            float(
                floor
            ),
            quantile_cut,
        )

    return {
        "threshold": threshold,
        "quantile_cut": quantile_cut,
        "strong": (
            np.isfinite(
                values
            )
            & (
                values
                >= threshold
            )
        ),
        "percentile": percentile_rank(
            values,
            mask=mask,
        ),
        "n_reference_cells": int(
            finite.sum()
        ),
    }


def numeric_obs(
    adata: ad.AnnData,
    column: str,
) -> np.ndarray:
    if column not in adata.obs.columns:
        raise KeyError(
            f"Missing required score column: {column}"
        )

    return (
        pd.to_numeric(
            adata.obs[
                column
            ],
            errors="coerce",
        )
        .fillna(0)
        .to_numpy(
            dtype=np.float32
        )
    )

In [7]:
# ---------------------------------------------------------------------
# Inclusive T-cell and Treg rescue hierarchy
# ---------------------------------------------------------------------
def apply_inclusive_rescue(
    adata: ad.AnnData,
    sample: str,
) -> tuple[
    pd.DataFrame,
    dict,
]:
    threshold_rows = []

    broad_scores = {
        key: numeric_obs(
            adata,
            column,
        )
        for key, column in (
            BROAD_SCORE_COLUMNS.items()
        )
    }
    broad_percentiles = {
        key: percentile_rank(
            values
        )
        for key, values in (
            broad_scores.items()
        )
    }

    for key, values in (
        broad_scores.items()
    ):
        adata.obs[
            f"rescue_{key}_score"
        ] = values
        adata.obs[
            f"rescue_{key}_percentile"
        ] = broad_percentiles[
            key
        ]

    T_score = broad_scores[
        "T_core"
    ]
    T_percentile = (
        broad_percentiles[
            "T_core"
        ]
    )
    non_T_percentiles = (
        np.column_stack(
            [
                broad_percentiles[
                    key
                ]
                for key in (
                    NON_T_BROAD_KEYS
                )
            ]
        )
    )
    max_non_T_percentile = (
        np.nanmax(
            non_T_percentiles,
            axis=1,
        )
    )

    primary_T_threshold = (
        quantile_floor_threshold(
            T_score,
            quantile=(
                PRIMARY_T_CORE_QUANTILE
            ),
            floor=(
                T_CORE_ABSOLUTE_FLOOR
            ),
        )
    )
    exploratory_T_threshold = (
        quantile_floor_threshold(
            T_score,
            quantile=(
                EXPLORATORY_T_CORE_QUANTILE
            ),
            floor=(
                T_CORE_ABSOLUTE_FLOOR
            ),
        )
    )

    for tier, result, quantile in (
        (
            "primary_T_core",
            primary_T_threshold,
            PRIMARY_T_CORE_QUANTILE,
        ),
        (
            "exploratory_T_core",
            exploratory_T_threshold,
            EXPLORATORY_T_CORE_QUANTILE,
        ),
    ):
        threshold_rows.append(
            {
                "sample": sample,
                "gate": tier,
                "score": "T_core",
                "quantile": quantile,
                "fixed_floor": (
                    T_CORE_ABSOLUTE_FLOOR
                ),
                "quantile_cut": result[
                    "quantile_cut"
                ],
                "final_threshold": result[
                    "threshold"
                ],
                "n_reference_cells": (
                    result[
                        "n_reference_cells"
                    ]
                ),
                "n_above_score_threshold": int(
                    result[
                        "strong"
                    ].sum()
                ),
            }
        )

    strict_T = (
        adata.obs[
            "fallback_T_lineage"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    raw_coherent = (
        adata.obs[
            "rescue_raw_T_coherent"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    raw_sensitive = (
        adata.obs[
            "rescue_raw_T_sensitive"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    reference_both = (
        adata.obs[
            "rescue_reference_both_T"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    reference_any = (
        adata.obs[
            "rescue_reference_any_T"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    primary_raw_rescue = (
        primary_T_threshold[
            "strong"
        ]
        & raw_coherent
        & (
            T_percentile
            >= (
                max_non_T_percentile
                - float(
                    PRIMARY_T_VS_NON_T_PERCENTILE_ALLOWANCE
                )
            )
        )
    )
    primary_reference_rescue = (
        primary_T_threshold[
            "strong"
        ]
        & reference_both
        & raw_sensitive
        & (
            T_percentile
            >= (
                max_non_T_percentile
                - float(
                    PRIMARY_REFERENCE_T_ALLOWANCE
                )
            )
        )
    )

    primary_T = (
        strict_T
        | primary_raw_rescue
        | primary_reference_rescue
    )

    exploratory_raw_rescue = (
        exploratory_T_threshold[
            "strong"
        ]
        & raw_coherent
    )
    exploratory_reference_rescue = (
        exploratory_T_threshold[
            "strong"
        ]
        & reference_any
        & raw_sensitive
    )

    # FOXP3 is not sufficient by itself; it can only rescue a T candidate when
    # a mildly high T-core score and CD4-compatible corrected signal coexist.
    CD4_score_all = numeric_obs(
        adata,
        SUBTYPE_SCORE_COLUMNS[
            "CD4"
        ],
    )
    CD4_percentile_all = percentile_rank(
        CD4_score_all
    )
    exploratory_FOXP3_rescue = (
        exploratory_T_threshold[
            "strong"
        ]
        & adata.obs[
            "rescue_raw_FOXP3_detected"
        ].to_numpy(
            dtype=bool
        )
        & (
            CD4_percentile_all
            >= float(
                CD4_COMPATIBILITY_PERCENTILE
            )
        )
        & (
            raw_sensitive
            | reference_any
        )
    )

    exploratory_T = (
        primary_T
        | exploratory_raw_rescue
        | exploratory_reference_rescue
        | exploratory_FOXP3_rescue
    )

    rescue_tier = np.full(
        adata.n_obs,
        "not_T_candidate",
        dtype=object,
    )
    rescue_tier[
        exploratory_T
    ] = "exploratory_rescue"
    rescue_tier[
        primary_T
    ] = "primary_rescue"
    rescue_tier[
        strict_T
    ] = "strict"

    T_mixing_status = np.full(
        adata.n_obs,
        "not_T_candidate",
        dtype=object,
    )
    T_mixing_status[
        exploratory_T
        & (
            T_percentile
            >= (
                max_non_T_percentile
                - 0.10
            )
        )
    ] = "T_supported_clean_or_balanced"
    T_mixing_status[
        exploratory_T
        & (
            T_percentile
            < (
                max_non_T_percentile
                - 0.10
            )
        )
    ] = "T_supported_spatially_mixed"

    adata.obs[
        "rescue_T_tier"
    ] = pd.Categorical(
        rescue_tier,
        categories=[
            "strict",
            "primary_rescue",
            "exploratory_rescue",
            "not_T_candidate",
        ],
    )
    adata.obs[
        "rescue_T_primary"
    ] = primary_T
    adata.obs[
        "rescue_T_exploratory"
    ] = exploratory_T
    adata.obs[
        "rescue_T_mixing_status"
    ] = pd.Categorical(
        T_mixing_status
    )
    adata.obs[
        "rescue_T_percentile_minus_max_nonT"
    ] = (
        T_percentile
        - max_non_T_percentile
    ).astype(
        np.float32
    )

    # --------------------------------------------------------------
    # Treg and CD4/CD8 subtyping.
    # --------------------------------------------------------------
    subtype_reference_primary = (
        primary_T
        if primary_T.sum() >= 20
        else exploratory_T
    )
    if subtype_reference_primary.sum() < 20:
        subtype_reference_primary = np.ones(
            adata.n_obs,
            dtype=bool,
        )

    subtype_reference_exploratory = (
        exploratory_T
        if exploratory_T.sum() >= 20
        else subtype_reference_primary
    )

    subtype_scores = {
        key: numeric_obs(
            adata,
            column,
        )
        for key, column in (
            SUBTYPE_SCORE_COLUMNS.items()
        )
    }

    primary_percentiles = {
        key: percentile_rank(
            values,
            mask=(
                subtype_reference_primary
            ),
        )
        for key, values in (
            subtype_scores.items()
        )
    }
    exploratory_percentiles = {
        key: percentile_rank(
            values,
            mask=(
                subtype_reference_exploratory
            ),
        )
        for key, values in (
            subtype_scores.items()
        )
    }

    for key, values in (
        subtype_scores.items()
    ):
        adata.obs[
            f"rescue_{key}_subtype_score"
        ] = values
        adata.obs[
            f"rescue_{key}_primary_percentile"
        ] = primary_percentiles[
            key
        ]
        adata.obs[
            f"rescue_{key}_exploratory_percentile"
        ] = exploratory_percentiles[
            key
        ]

    supported_Treg_threshold = (
        quantile_floor_threshold(
            subtype_scores[
                "Treg"
            ],
            quantile=(
                SUPPORTED_TREG_QUANTILE
            ),
            floor=(
                TREG_ABSOLUTE_FLOOR
            ),
            mask=(
                subtype_reference_primary
            ),
        )
    )
    exploratory_Treg_threshold = (
        quantile_floor_threshold(
            subtype_scores[
                "Treg"
            ],
            quantile=(
                EXPLORATORY_TREG_QUANTILE
            ),
            floor=(
                TREG_ABSOLUTE_FLOOR
            ),
            mask=(
                subtype_reference_exploratory
            ),
        )
    )

    for tier, result, quantile in (
        (
            "supported_Treg",
            supported_Treg_threshold,
            SUPPORTED_TREG_QUANTILE,
        ),
        (
            "exploratory_Treg",
            exploratory_Treg_threshold,
            EXPLORATORY_TREG_QUANTILE,
        ),
    ):
        threshold_rows.append(
            {
                "sample": sample,
                "gate": tier,
                "score": "Treg_core",
                "quantile": quantile,
                "fixed_floor": (
                    TREG_ABSOLUTE_FLOOR
                ),
                "quantile_cut": result[
                    "quantile_cut"
                ],
                "final_threshold": result[
                    "threshold"
                ],
                "n_reference_cells": (
                    result[
                        "n_reference_cells"
                    ]
                ),
                "n_above_score_threshold": int(
                    result[
                        "strong"
                    ].sum()
                ),
            }
        )

    raw_Treg_anchor = (
        adata.obs[
            "rescue_raw_Treg_anchor"
        ].to_numpy(
            dtype=bool
        )
    )
    raw_FOXP3 = (
        adata.obs[
            "rescue_raw_FOXP3_detected"
        ].to_numpy(
            dtype=bool
        )
    )

    CD4_primary_pct = (
        primary_percentiles[
            "CD4"
        ]
    )
    CD8_primary_pct = (
        primary_percentiles[
            "CD8"
        ]
    )
    Treg_primary_pct = (
        primary_percentiles[
            "Treg"
        ]
    )
    NK_primary_pct = (
        primary_percentiles[
            "NK"
        ]
    )

    CD4_exploratory_pct = (
        exploratory_percentiles[
            "CD4"
        ]
    )
    CD8_exploratory_pct = (
        exploratory_percentiles[
            "CD8"
        ]
    )
    Treg_exploratory_pct = (
        exploratory_percentiles[
            "Treg"
        ]
    )

    supported_CD4_compatible = (
        CD4_primary_pct
        >= float(
            CD4_COMPATIBILITY_PERCENTILE
        )
    ) | (
        adata.obs[
            "rescue_raw_CD4_detected"
        ].to_numpy(
            dtype=bool
        )
    )
    supported_not_CD8_dominant = (
        Treg_primary_pct
        >= (
            CD8_primary_pct
            - float(
                SUPPORTED_TREG_VS_CD8_ALLOWANCE
            )
        )
    )

    Treg_supported = (
        primary_T
        & supported_Treg_threshold[
            "strong"
        ]
        & supported_CD4_compatible
        & supported_not_CD8_dominant
    )
    Treg_high_confidence = (
        Treg_supported
        & raw_Treg_anchor
    )

    exploratory_CD4_compatible = (
        CD4_exploratory_pct
        >= float(
            CD4_COMPATIBILITY_PERCENTILE
        )
    ) | (
        adata.obs[
            "rescue_raw_CD4_detected"
        ].to_numpy(
            dtype=bool
        )
    )
    exploratory_not_CD8_dominant = (
        Treg_exploratory_pct
        >= (
            CD8_exploratory_pct
            - float(
                EXPLORATORY_TREG_VS_CD8_ALLOWANCE
            )
        )
    )

    Treg_exploratory = (
        exploratory_T
        & exploratory_Treg_threshold[
            "strong"
        ]
        & exploratory_CD4_compatible
        & exploratory_not_CD8_dominant
        & (
            raw_FOXP3
            | raw_Treg_anchor
            | supported_Treg_threshold[
                "strong"
            ]
        )
    )

    adata.obs[
        "rescue_Treg_high_confidence"
    ] = Treg_high_confidence
    adata.obs[
        "rescue_Treg_supported"
    ] = Treg_supported
    adata.obs[
        "rescue_Treg_exploratory"
    ] = Treg_exploratory

    # --------------------------------------------------------------
    # CD4/CD8 labels. CD8B is not used.
    # --------------------------------------------------------------
    CD8_primary = (
        primary_T
        & ~Treg_supported
        & (
            CD8_primary_pct
            >= float(
                CD4_CD8_POSITIVE_PERCENTILE
            )
        )
        & (
            CD8_primary_pct
            >= (
                CD4_primary_pct
                + float(
                    CD4_CD8_DOMINANCE_MARGIN
                )
            )
        )
        & (
            CD8_primary_pct
            >= (
                NK_primary_pct
                - float(
                    CD8_VS_NK_ALLOWANCE
                )
            )
        )
    )
    CD4_primary = (
        primary_T
        & ~Treg_supported
        & ~CD8_primary
        & (
            CD4_primary_pct
            >= float(
                CD4_CD8_POSITIVE_PERCENTILE
            )
        )
        & (
            CD4_primary_pct
            >= (
                CD8_primary_pct
                + float(
                    CD4_CD8_DOMINANCE_MARGIN
                )
            )
        )
    )
    CD4_CD8_ambiguous_primary = (
        primary_T
        & ~Treg_supported
        & ~CD8_primary
        & ~CD4_primary
        & (
            CD4_primary_pct
            >= float(
                CD4_CD8_POSITIVE_PERCENTILE
            )
        )
        & (
            CD8_primary_pct
            >= float(
                CD4_CD8_POSITIVE_PERCENTILE
            )
        )
    )

    primary_subtype = np.full(
        adata.n_obs,
        "",
        dtype=object,
    )
    primary_subtype[
        primary_T
    ] = "Tcell:unspecified"
    primary_subtype[
        CD4_CD8_ambiguous_primary
    ] = "Tcell:CD4_CD8_ambiguous"
    primary_subtype[
        CD4_primary
    ] = "Tcell:CD4+"
    primary_subtype[
        CD8_primary
    ] = "Tcell:CD8+"
    primary_subtype[
        Treg_supported
    ] = "Tcell:Treg_supported"
    primary_subtype[
        Treg_high_confidence
    ] = "Tcell:Treg_high_confidence"

    # Exploratory label uses exploratory Treg but retains primary CD4/CD8 labels
    # where available.
    exploratory_subtype = np.full(
        adata.n_obs,
        "",
        dtype=object,
    )
    exploratory_subtype[
        exploratory_T
    ] = "Tcell:unspecified"
    exploratory_subtype[
        primary_T
    ] = primary_subtype[
        primary_T
    ]
    exploratory_subtype[
        Treg_exploratory
    ] = "Tcell:Treg_exploratory"
    exploratory_subtype[
        Treg_supported
    ] = "Tcell:Treg_supported"
    exploratory_subtype[
        Treg_high_confidence
    ] = "Tcell:Treg_high_confidence"

    adata.obs[
        "rescue_T_subtype_primary"
    ] = pd.Categorical(
        primary_subtype
    )
    adata.obs[
        "rescue_T_subtype_exploratory"
    ] = pd.Categorical(
        exploratory_subtype
    )

    old_broad = (
        adata.obs[
            "fallback_level1_lineage"
        ]
        .astype(str)
        .to_numpy(
            dtype=object
        )
    )

    primary_cell_type = old_broad.copy()
    primary_cell_type[
        primary_T
    ] = primary_subtype[
        primary_T
    ]

    exploratory_cell_type = old_broad.copy()
    exploratory_cell_type[
        exploratory_T
    ] = exploratory_subtype[
        exploratory_T
    ]

    adata.obs[
        "rescue_cell_type_primary"
    ] = pd.Categorical(
        primary_cell_type
    )
    adata.obs[
        "rescue_cell_type_exploratory"
    ] = pd.Categorical(
        exploratory_cell_type
    )

    threshold_table = pd.DataFrame(
        threshold_rows
    )

    summary = {
        "n_cells": int(
            adata.n_obs
        ),
        "n_strict_T": int(
            strict_T.sum()
        ),
        "n_primary_T": int(
            primary_T.sum()
        ),
        "n_exploratory_T": int(
            exploratory_T.sum()
        ),
        "n_primary_new_vs_strict": int(
            (
                primary_T
                & ~strict_T
            ).sum()
        ),
        "n_exploratory_new_vs_primary": int(
            (
                exploratory_T
                & ~primary_T
            ).sum()
        ),
        "n_raw_T_coherent": int(
            raw_coherent.sum()
        ),
        "n_reference_both_T": int(
            reference_both.sum()
        ),
        "n_Treg_high_confidence": int(
            Treg_high_confidence.sum()
        ),
        "n_Treg_supported": int(
            Treg_supported.sum()
        ),
        "n_Treg_exploratory": int(
            Treg_exploratory.sum()
        ),
        "n_raw_FOXP3_positive": int(
            raw_FOXP3.sum()
        ),
        "n_raw_FOXP3_in_primary_T": int(
            (
                raw_FOXP3
                & primary_T
            ).sum()
        ),
        "n_raw_FOXP3_in_exploratory_T": int(
            (
                raw_FOXP3
                & exploratory_T
            ).sum()
        ),
        "primary_T_spatially_mixed": int(
            (
                primary_T
                & (
                    T_mixing_status
                    == "T_supported_spatially_mixed"
                )
            ).sum()
        ),
    }

    return (
        threshold_table,
        summary,
    )

In [8]:
# ---------------------------------------------------------------------
# Plots and output tables
# ---------------------------------------------------------------------
def plotting_indices(
    n_obs: int,
) -> np.ndarray:
    if n_obs <= int(
        PLOT_MAX_CELLS
    ):
        return np.arange(
            n_obs
        )

    rng = np.random.default_rng(
        RANDOM_STATE
    )
    return np.sort(
        rng.choice(
            n_obs,
            size=int(
                PLOT_MAX_CELLS
            ),
            replace=False,
        )
    )


def save_T_rescue_plot(
    adata: ad.AnnData,
    sample: str,
    path: Path,
) -> None:
    indices = plotting_indices(
        adata.n_obs
    )

    T_pct = pd.to_numeric(
        adata.obs[
            "rescue_T_core_percentile"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)
    max_non_T = np.nanmax(
        np.column_stack(
            [
                pd.to_numeric(
                    adata.obs[
                        f"rescue_{key}_percentile"
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float)
                for key in NON_T_BROAD_KEYS
            ]
        ),
        axis=1,
    )

    tier = (
        adata.obs[
            "rescue_T_tier"
        ]
        .astype(str)
        .to_numpy()
    )
    raw_coherent = (
        adata.obs[
            "rescue_raw_T_coherent"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6),
    )

    axes[0].scatter(
        max_non_T[
            indices
        ],
        T_pct[
            indices
        ],
        s=4,
        linewidths=0,
        alpha=0.25,
        rasterized=True,
    )

    for label in (
        "strict",
        "primary_rescue",
        "exploratory_rescue",
    ):
        mask = (
            tier == label
        )
        if mask.any():
            axes[0].scatter(
                max_non_T[
                    mask
                ],
                T_pct[
                    mask
                ],
                s=8,
                linewidths=0,
                alpha=0.75,
                label=label,
                rasterized=True,
            )

    x = np.linspace(
        0,
        1,
        200,
    )
    axes[0].plot(
        x,
        x
        - float(
            PRIMARY_T_VS_NON_T_PERCENTILE_ALLOWANCE
        ),
        linestyle="--",
        label="primary raw-T allowance",
    )
    axes[0].set_xlabel(
        "Maximum competing lineage percentile"
    )
    axes[0].set_ylabel(
        "T-core percentile"
    )
    axes[0].set_title(
        "Inclusive T-cell rescue"
    )
    axes[0].legend(
        fontsize=8,
        frameon=False,
    )

    funnel = pd.Series(
        {
            "all cells": (
                adata.n_obs
            ),
            "raw T coherent": int(
                raw_coherent.sum()
            ),
            "strict T": int(
                (
                    tier
                    == "strict"
                ).sum()
            ),
            "primary T total": int(
                adata.obs[
                    "rescue_T_primary"
                ].sum()
            ),
            "exploratory T total": int(
                adata.obs[
                    "rescue_T_exploratory"
                ].sum()
            ),
        }
    )
    funnel.plot(
        kind="bar",
        ax=axes[1],
    )
    axes[1].set_ylabel(
        "Cells"
    )
    axes[1].set_title(
        "T-lineage evidence tiers"
    )
    axes[1].tick_params(
        axis="x",
        rotation=30,
    )

    fig.suptitle(
        f"{sample}: strict and inclusive T-cell rescue"
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_Treg_rescue_plot(
    adata: ad.AnnData,
    sample: str,
    path: Path,
) -> None:
    Treg_pct = pd.to_numeric(
        adata.obs[
            "rescue_Treg_exploratory_percentile"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)
    CD8_pct = pd.to_numeric(
        adata.obs[
            "rescue_CD8_exploratory_percentile"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    exploratory_T = (
        adata.obs[
            "rescue_T_exploratory"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    FOXP3 = (
        adata.obs[
            "rescue_raw_FOXP3_detected"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    high = (
        adata.obs[
            "rescue_Treg_high_confidence"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    supported = (
        adata.obs[
            "rescue_Treg_supported"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    exploratory = (
        adata.obs[
            "rescue_Treg_exploratory"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6),
    )

    base = (
        exploratory_T
        & np.isfinite(
            Treg_pct
        )
        & np.isfinite(
            CD8_pct
        )
    )
    axes[0].scatter(
        CD8_pct[
            base
        ],
        Treg_pct[
            base
        ],
        s=6,
        linewidths=0,
        alpha=0.30,
        rasterized=True,
    )
    if FOXP3.any():
        axes[0].scatter(
            CD8_pct[
                base
                & FOXP3
            ],
            Treg_pct[
                base
                & FOXP3
            ],
            marker="x",
            s=25,
            label="raw FOXP3+",
            rasterized=True,
        )
    if exploratory.any():
        axes[0].scatter(
            CD8_pct[
                exploratory
            ],
            Treg_pct[
                exploratory
            ],
            facecolors="none",
            s=35,
            label="exploratory Treg",
            rasterized=True,
        )
    if supported.any():
        axes[0].scatter(
            CD8_pct[
                supported
            ],
            Treg_pct[
                supported
            ],
            marker="s",
            s=35,
            label="supported Treg",
            rasterized=True,
        )
    if high.any():
        axes[0].scatter(
            CD8_pct[
                high
            ],
            Treg_pct[
                high
            ],
            marker="*",
            s=80,
            label="high-confidence Treg",
            rasterized=True,
        )

    x = np.linspace(
        0,
        1,
        200,
    )
    axes[0].plot(
        x,
        x
        - float(
            EXPLORATORY_TREG_VS_CD8_ALLOWANCE
        ),
        linestyle="--",
        label="exploratory Treg/CD8 allowance",
    )
    axes[0].set_xlabel(
        "CD8-noCD8B percentile"
    )
    axes[0].set_ylabel(
        "Treg-core percentile"
    )
    axes[0].set_title(
        "Treg versus CD8 evidence"
    )
    axes[0].legend(
        fontsize=7,
        frameon=False,
    )

    funnel = pd.Series(
        {
            "raw FOXP3+ all": int(
                FOXP3.sum()
            ),
            "raw FOXP3+ exploratory T": int(
                (
                    FOXP3
                    & exploratory_T
                ).sum()
            ),
            "exploratory Treg": int(
                exploratory.sum()
            ),
            "supported Treg": int(
                supported.sum()
            ),
            "high-confidence Treg": int(
                high.sum()
            ),
        }
    )
    funnel.plot(
        kind="bar",
        ax=axes[1],
    )
    axes[1].set_ylabel(
        "Cells"
    )
    axes[1].set_title(
        "Treg evidence tiers"
    )
    axes[1].tick_params(
        axis="x",
        rotation=35,
    )

    fig.suptitle(
        f"{sample}: inclusive Treg rescue"
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_spatial_plot(
    adata: ad.AnnData,
    sample: str,
    path: Path,
) -> str | None:
    key = choose_spatial_key(
        adata
    )
    if key is None:
        return None

    coordinates = np.asarray(
        adata.obsm[
            key
        ],
        dtype=float,
    )
    indices = plotting_indices(
        adata.n_obs
    )

    labels = np.full(
        adata.n_obs,
        "not_T",
        dtype=object,
    )
    labels[
        adata.obs[
            "rescue_T_exploratory"
        ].to_numpy(
            dtype=bool
        )
    ] = "exploratory_T"
    labels[
        adata.obs[
            "rescue_T_primary"
        ].to_numpy(
            dtype=bool
        )
    ] = "primary_T"
    labels[
        adata.obs[
            "fallback_T_lineage"
        ].to_numpy(
            dtype=bool
        )
    ] = "strict_T"
    labels[
        adata.obs[
            "rescue_Treg_exploratory"
        ].to_numpy(
            dtype=bool
        )
    ] = "exploratory_Treg"
    labels[
        adata.obs[
            "rescue_Treg_supported"
        ].to_numpy(
            dtype=bool
        )
    ] = "supported_Treg"
    labels[
        adata.obs[
            "rescue_Treg_high_confidence"
        ].to_numpy(
            dtype=bool
        )
    ] = "high_confidence_Treg"

    plot_labels = labels[
        indices
    ]
    categories = [
        category
        for category in (
            "not_T",
            "exploratory_T",
            "primary_T",
            "strict_T",
            "exploratory_Treg",
            "supported_Treg",
            "high_confidence_Treg",
        )
        if (
            plot_labels
            == category
        ).any()
    ]

    fig, ax = plt.subplots(
        figsize=(9, 8),
    )
    for category in categories:
        mask = (
            plot_labels
            == category
        )
        ax.scatter(
            coordinates[
                indices[
                    mask
                ],
                0,
            ],
            coordinates[
                indices[
                    mask
                ],
                1,
            ],
            s=(
                5
                if "Treg"
                in category
                else 2
            ),
            linewidths=0,
            alpha=(
                0.90
                if "Treg"
                in category
                else 0.65
            ),
            label=category,
            rasterized=True,
        )

    ax.set_title(
        f"{sample}: strict and inclusive T/Treg candidates"
    )
    ax.set_xlabel(
        "Spatial x"
    )
    ax.set_ylabel(
        "Spatial y"
    )
    ax.set_aspect(
        "equal"
    )
    ax.invert_yaxis()
    ax.legend(
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
        fontsize=7,
        frameon=False,
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    return key

In [9]:
# ---------------------------------------------------------------------
# Process one sample
# ---------------------------------------------------------------------
def process_sample(
    sample: str,
) -> tuple[
    dict,
    ad.AnnData,
]:
    paths = paths_for_sample(
        sample
    )

    if not paths[
        "source_h5ad"
    ].exists():
        raise FileNotFoundError(
            paths[
                "source_h5ad"
            ]
        )

    print(
        "\n"
        + "=" * 90
    )
    print(
        "Inclusive T/Treg rescue:",
        sample,
    )

    adata = ad.read_h5ad(
        paths[
            "source_h5ad"
        ]
    )
    adata.obs_names = (
        adata.obs_names.astype(str)
    )

    for column, value in (
        SAMPLE_INFO[
            sample
        ].items()
    ):
        adata.obs[
            column
        ] = value
    adata.obs[
        "sample"
    ] = sample

    raw_mapping = add_raw_evidence(
        adata
    )
    reference_info = (
        add_reference_T_evidence(
            adata
        )
    )
    thresholds, rescue_summary = (
        apply_inclusive_rescue(
            adata,
            sample,
        )
    )

    thresholds.to_csv(
        paths[
            "thresholds"
        ],
        index=False,
    )

    strict_T = (
        adata.obs[
            "fallback_T_lineage"
        ].to_numpy(
            dtype=bool
        )
    )
    primary_T = (
        adata.obs[
            "rescue_T_primary"
        ].to_numpy(
            dtype=bool
        )
    )
    exploratory_T = (
        adata.obs[
            "rescue_T_exploratory"
        ].to_numpy(
            dtype=bool
        )
    )
    Treg_high = (
        adata.obs[
            "rescue_Treg_high_confidence"
        ].to_numpy(
            dtype=bool
        )
    )
    Treg_supported = (
        adata.obs[
            "rescue_Treg_supported"
        ].to_numpy(
            dtype=bool
        )
    )
    Treg_exploratory = (
        adata.obs[
            "rescue_Treg_exploratory"
        ].to_numpy(
            dtype=bool
        )
    )

    count_rows = [
        {
            "sample": sample,
            "metric": "all_cells",
            "n_cells": int(
                adata.n_obs
            ),
        },
        {
            "sample": sample,
            "metric": "strict_T",
            "n_cells": int(
                strict_T.sum()
            ),
        },
        {
            "sample": sample,
            "metric": "primary_T",
            "n_cells": int(
                primary_T.sum()
            ),
        },
        {
            "sample": sample,
            "metric": "exploratory_T",
            "n_cells": int(
                exploratory_T.sum()
            ),
        },
        {
            "sample": sample,
            "metric": "strict_Treg",
            "n_cells": int(
                adata.obs[
                    "fallback_Treg_signature"
                ].sum()
            ),
        },
        {
            "sample": sample,
            "metric": "Treg_high_confidence",
            "n_cells": int(
                Treg_high.sum()
            ),
        },
        {
            "sample": sample,
            "metric": "Treg_supported",
            "n_cells": int(
                Treg_supported.sum()
            ),
        },
        {
            "sample": sample,
            "metric": "Treg_exploratory",
            "n_cells": int(
                Treg_exploratory.sum()
            ),
        },
        {
            "sample": sample,
            "metric": "raw_FOXP3_positive",
            "n_cells": int(
                adata.obs[
                    "rescue_raw_FOXP3_detected"
                ].sum()
            ),
        },
        {
            "sample": sample,
            "metric": "raw_T_coherent",
            "n_cells": int(
                adata.obs[
                    "rescue_raw_T_coherent"
                ].sum()
            ),
        },
    ]
    counts = pd.DataFrame(
        count_rows
    )
    counts.to_csv(
        paths[
            "counts"
        ],
        index=False,
    )

    candidate_ids = pd.DataFrame(
        {
            "cell_id": (
                adata.obs_names.astype(str)
            ),
            "strict_T": strict_T,
            "primary_T": primary_T,
            "exploratory_T": exploratory_T,
            "Treg_high_confidence": Treg_high,
            "Treg_supported": Treg_supported,
            "Treg_exploratory": Treg_exploratory,
            "raw_T_coherent": (
                adata.obs[
                    "rescue_raw_T_coherent"
                ].to_numpy(
                    dtype=bool
                )
            ),
            "raw_FOXP3": (
                adata.obs[
                    "rescue_raw_FOXP3_detected"
                ].to_numpy(
                    dtype=bool
                )
            ),
            "T_mixing_status": (
                adata.obs[
                    "rescue_T_mixing_status"
                ].astype(str).to_numpy()
            ),
            "primary_subtype": (
                adata.obs[
                    "rescue_T_subtype_primary"
                ].astype(str).to_numpy()
            ),
            "exploratory_subtype": (
                adata.obs[
                    "rescue_T_subtype_exploratory"
                ].astype(str).to_numpy()
            ),
        }
    )
    candidate_ids.to_csv(
        paths[
            "candidate_ids"
        ],
        index=False,
    )

    output_columns = [
        column
        for column in adata.obs.columns
        if (
            column.startswith(
                "rescue_"
            )
            or column.startswith(
                "fallback_"
            )
            or column
            in {
                "sample",
                "patient",
                "cancer_type",
                "biopsy_stage",
            }
        )
    ]
    metadata = (
        adata.obs[
            output_columns
        ]
        .copy()
    )
    metadata.insert(
        0,
        "cell_id",
        adata.obs_names.astype(str),
    )

    if WRITE_METADATA_PARQUET:
        metadata.to_parquet(
            paths[
                "metadata"
            ],
            index=False,
        )

    spatial_key = choose_spatial_key(
        adata
    )
    if (
        WRITE_SPATIAL_PARQUET
        and spatial_key
        is not None
    ):
        coordinates = np.asarray(
            adata.obsm[
                spatial_key
            ],
            dtype=np.float32,
        )
        spatial = metadata.copy()
        spatial[
            "spatial_x"
        ] = coordinates[
            :,
            0,
        ]
        spatial[
            "spatial_y"
        ] = coordinates[
            :,
            1,
        ]
        spatial[
            "spatial_source"
        ] = spatial_key
        spatial.to_parquet(
            paths[
                "spatial"
            ],
            index=False,
        )

    save_T_rescue_plot(
        adata,
        sample,
        paths["figures"]
        / f"{sample}_inclusive_T_rescue.png",
    )
    save_Treg_rescue_plot(
        adata,
        sample,
        paths["figures"]
        / f"{sample}_inclusive_Treg_rescue.png",
    )
    spatial_key = save_spatial_plot(
        adata,
        sample,
        paths["figures"]
        / f"{sample}_inclusive_T_Treg_spatial.png",
    )

    # Compact scalar/JSON provenance only.
    adata.uns[
        "final_inclusive_T_Treg_rescue"
    ] = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "source_h5ad": str(
            paths[
                "source_h5ad"
            ]
        ),
        "threshold_table": str(
            paths[
                "thresholds"
            ]
        ),
        "CD8B_used": False,
        "primary_T_core_quantile": float(
            PRIMARY_T_CORE_QUANTILE
        ),
        "exploratory_T_core_quantile": float(
            EXPLORATORY_T_CORE_QUANTILE
        ),
        "supported_Treg_quantile": float(
            SUPPORTED_TREG_QUANTILE
        ),
        "exploratory_Treg_quantile": float(
            EXPLORATORY_TREG_QUANTILE
        ),
        "raw_gene_mapping_json": json.dumps(
            raw_mapping,
            default=str,
            sort_keys=True,
        ),
        "reference_info_json": json.dumps(
            reference_info,
            default=str,
            sort_keys=True,
        ),
        "bin_reference_json": json.dumps(
            BIN_REFERENCE.get(
                sample,
                {}
            ),
            default=str,
            sort_keys=True,
        ),
    }

    if WRITE_FULL_ANNOTATED_H5AD:
        safe_write_h5ad(
            adata,
            paths[
                "annotated_h5ad"
            ],
            compression=(
                H5AD_COMPRESSION
            ),
        )

    if WRITE_PRIMARY_T_H5AD:
        primary = adata[
            primary_T
        ].copy()
        primary.uns[
            "final_inclusive_T_Treg_rescue"
        ][
            "subset_interpretation"
        ] = (
            "strict plus primary rescued T cells"
        )
        safe_write_h5ad(
            primary,
            paths[
                "primary_T_h5ad"
            ],
            compression=(
                H5AD_COMPRESSION
            ),
        )
        del primary
        gc.collect()

    if WRITE_EXPLORATORY_T_H5AD:
        exploratory = adata[
            exploratory_T
        ].copy()
        exploratory.uns[
            "final_inclusive_T_Treg_rescue"
        ][
            "subset_interpretation"
        ] = (
            "strict, primary, and exploratory rescued T cells"
        )
        safe_write_h5ad(
            exploratory,
            paths[
                "exploratory_T_h5ad"
            ],
            compression=(
                H5AD_COMPRESSION
            ),
        )
        del exploratory
        gc.collect()

    summary = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "sample": sample,
        **SAMPLE_INFO[
            sample
        ],
        **rescue_summary,
        **BIN_REFERENCE.get(
            sample,
            {}
        ),
        "raw_gene_mapping": (
            raw_mapping
        ),
        "reference_info": (
            reference_info
        ),
        "spatial_key": (
            spatial_key
        ),
        "annotated_h5ad": str(
            paths[
                "annotated_h5ad"
            ]
        )
        if WRITE_FULL_ANNOTATED_H5AD
        else None,
        "primary_T_h5ad": str(
            paths[
                "primary_T_h5ad"
            ]
        )
        if WRITE_PRIMARY_T_H5AD
        else None,
        "exploratory_T_h5ad": str(
            paths[
                "exploratory_T_h5ad"
            ]
        )
        if WRITE_EXPLORATORY_T_H5AD
        else None,
    }
    write_json(
        summary,
        paths[
            "summary"
        ],
    )

    print(
        sample,
        {
            "strict_T": (
                summary[
                    "n_strict_T"
                ]
            ),
            "primary_T": (
                summary[
                    "n_primary_T"
                ]
            ),
            "exploratory_T": (
                summary[
                    "n_exploratory_T"
                ]
            ),
            "Treg_high": (
                summary[
                    "n_Treg_high_confidence"
                ]
            ),
            "Treg_supported": (
                summary[
                    "n_Treg_supported"
                ]
            ),
            "Treg_exploratory": (
                summary[
                    "n_Treg_exploratory"
                ]
            ),
            "raw_FOXP3": (
                summary[
                    "n_raw_FOXP3_positive"
                ]
            ),
        },
    )

    return (
        summary,
        adata,
    )

In [10]:
# ---------------------------------------------------------------------
# Run all samples
# ---------------------------------------------------------------------
import traceback

results = {}
failures = {}

for sample in SECTION_NAMES:
    try:
        summary, adata = process_sample(
            sample
        )
        results[
            sample
        ] = summary

        del adata
        gc.collect()

    except Exception as exc:
        failures[
            sample
        ] = (
            f"{type(exc).__name__}: {exc}"
        )
        print(
            f"[FAILED] {sample}: "
            f"{type(exc).__name__}: {exc}"
        )
        traceback.print_exc(
            limit=12
        )

        if not CONTINUE_ON_ERROR:
            raise

    finally:
        plt.close(
            "all"
        )
        gc.collect()

write_json(
    results,
    OUTPUT_ROOT
    / "all_sample_inclusive_rescue_results.json",
)
write_json(
    failures,
    OUTPUT_ROOT
    / "all_sample_inclusive_rescue_failures.json",
)

print(
    "Completed:",
    sorted(
        results
    ),
)
print(
    "Failures:",
    json.dumps(
        failures,
        indent=2,
    ),
)


Inclusive T/Treg rescue: Screen_39_21


/tmp/ipykernel_88104/2166877664.py:204: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(
/tmp/ipykernel_88104/2166877664.py:204: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/Screen_39_21/Screen_39_21_inclusive_rescue_annotated.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/Screen_39_21/Screen_39_21_primary_rescued_Tcells.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/Screen_39_21/Screen_39_21_exploratory_rescued_Tcells.h5ad
Screen_39_21 {'strict_T': 1114, 'primary_T': 2039, 'exploratory_T': 2687, 'Treg_high': 7, 'Treg_supported': 92, 'Treg_exploratory': 117, 'raw_FOXP3': 165}

Inclusive T/Treg rescue: C2D15_39_21


/tmp/ipykernel_88104/2166877664.py:204: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(
/tmp/ipykernel_88104/2166877664.py:204: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_39_21/C2D15_39_21_inclusive_rescue_annotated.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_39_21/C2D15_39_21_primary_rescued_Tcells.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_39_21/C2D15_39_21_exploratory_rescued_Tcells.h5ad
C2D15_39_21 {'strict_T': 1079, 'primary_T': 1172, 'exploratory_T': 1339, 'Treg_high': 1, 'Treg_supported': 22, 'Treg_exploratory': 25, 'raw_FOXP3': 80}

Inclusive T/Treg rescue: Screen_17_26


...

Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_30_16/C2D15_30_16_inclusive_rescue_annotated.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_30_16/C2D15_30_16_primary_rescued_Tcells.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_30_16/C2D15_30_16_exploratory_rescued_Tcells.h5ad
C2D15_30_16 {'strict_T': 0, 'primary_T': 437, 'exploratory_T': 1241, 'Treg_high': 0, 'Treg_supported': 20, 'Treg_exploratory': 43, 'raw_FOXP3': 58}

Inclusive T/Treg rescue: Screen_23_25


/tmp/ipykernel_88104/2166877664.py:204: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(
/tmp/ipykernel_88104/2166877664.py:204: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/Screen_23_25/Screen_23_25_inclusive_rescue_annotated.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/Screen_23_25/Screen_23_25_primary_rescued_Tcells.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/Screen_23_25/Screen_23_25_exploratory_rescued_Tcells.h5ad
Screen_23_25 {'strict_T': 4, 'primary_T': 99, 'exploratory_T': 231, 'Treg_high': 0, 'Treg_supported': 5, 'Treg_exploratory': 10, 'raw_FOXP3': 16}

Inclusive T/Treg rescue: C2D15_23_25


/tmp/ipykernel_88104/2166877664.py:204: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(
/tmp/ipykernel_88104/2166877664.py:204: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_23_25/C2D15_23_25_inclusive_rescue_annotated.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_23_25/C2D15_23_25_primary_rescued_Tcells.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/C2D15_23_25/C2D15_23_25_exploratory_rescued_Tcells.h5ad
C2D15_23_25 {'strict_T': 109, 'primary_T': 115, 'exploratory_T': 126, 'Treg_high': 0, 'Treg_supported': 4, 'Treg_exploratory': 4, 'raw_FOXP3': 3}
Completed: ['C2D15_16_22', 'C2D15_17_26', 'C2D15_18_23', 'C2D15_23_25', 'C2D15_30_16', 'C2D15_39_21', 'Screen_16_22', 'Screen_17_26', 'Screen_18_23', 'Screen_23_25', 'Screen_30_16', 'Screen_39_21']
Failures: {}


In [11]:
# ---------------------------------------------------------------------
# Cross-sample and paired Screen/C2D15 summaries
# ---------------------------------------------------------------------
summary = pd.DataFrame(
    [
        result
        for result in (
            results.values()
        )
    ]
)

if len(
    summary
):
    preferred_columns = [
        "sample",
        "patient",
        "cancer_type",
        "biopsy_stage",
        "n_cells",
        "n_strict_T",
        "n_primary_T",
        "n_exploratory_T",
        "n_raw_T_coherent",
        "n_Treg_high_confidence",
        "n_Treg_supported",
        "n_Treg_exploratory",
        "n_raw_FOXP3_positive",
        "n_raw_FOXP3_in_primary_T",
        "n_raw_FOXP3_in_exploratory_T",
        "FOXP3_positive_bins",
        "FOXP3_cells_bins_div3",
        "CD8A_positive_bins",
        "CD8A_cells_bins_div3",
        "primary_T_spatially_mixed",
    ]
    preferred_columns = [
        column
        for column in (
            preferred_columns
        )
        if column in summary.columns
    ]
    summary = summary[
        preferred_columns
    ].copy()

    summary.to_csv(
        OUTPUT_ROOT
        / "inclusive_rescue_summary_by_sample.csv",
        index=False,
    )

    # Fractions and audit ratios are descriptive only.
    summary[
        "supported_Treg_fraction_of_primary_T"
    ] = (
        summary[
            "n_Treg_supported"
        ]
        / summary[
            "n_primary_T"
        ].replace(
            0,
            np.nan,
        )
    )
    summary[
        "exploratory_Treg_fraction_of_exploratory_T"
    ] = (
        summary[
            "n_Treg_exploratory"
        ]
        / summary[
            "n_exploratory_T"
        ].replace(
            0,
            np.nan,
        )
    )
    summary[
        "supported_Treg_per_FOXP3_bin_scale"
    ] = (
        summary[
            "n_Treg_supported"
        ]
        / summary[
            "FOXP3_cells_bins_div3"
        ].replace(
            0,
            np.nan,
        )
    )
    summary[
        "exploratory_Treg_per_FOXP3_bin_scale"
    ] = (
        summary[
            "n_Treg_exploratory"
        ]
        / summary[
            "FOXP3_cells_bins_div3"
        ].replace(
            0,
            np.nan,
        )
    )

    summary.to_csv(
        OUTPUT_ROOT
        / "inclusive_rescue_summary_by_sample_with_fractions.csv",
        index=False,
    )

    paired_rows = []
    for patient, frame in (
        summary.groupby(
            "patient",
            observed=True,
        )
    ):
        by_stage = (
            frame
            .set_index(
                "biopsy_stage"
            )
        )
        if not {
            "Screen",
            "C2D15",
        }.issubset(
            by_stage.index
        ):
            continue

        row = {
            "patient": patient,
            "cancer_type": (
                frame[
                    "cancer_type"
                ].iloc[0]
            ),
        }

        for metric in (
            "n_strict_T",
            "n_primary_T",
            "n_exploratory_T",
            "n_Treg_high_confidence",
            "n_Treg_supported",
            "n_Treg_exploratory",
            "n_raw_FOXP3_positive",
            "FOXP3_positive_bins",
            "FOXP3_cells_bins_div3",
        ):
            screen_value = float(
                by_stage.loc[
                    "Screen",
                    metric,
                ]
            )
            C2D15_value = float(
                by_stage.loc[
                    "C2D15",
                    metric,
                ]
            )

            row[
                f"Screen_{metric}"
            ] = screen_value
            row[
                f"C2D15_{metric}"
            ] = C2D15_value
            row[
                f"delta_C2D15_minus_Screen_{metric}"
            ] = (
                C2D15_value
                - screen_value
            )

        paired_rows.append(
            row
        )

    paired = pd.DataFrame(
        paired_rows
    )
    paired.to_csv(
        OUTPUT_ROOT
        / "paired_Screen_C2D15_inclusive_rescue_summary.csv",
        index=False,
    )

    # Cross-sample count comparison.
    count_columns = [
        "n_strict_T",
        "n_primary_T",
        "n_exploratory_T",
        "n_Treg_high_confidence",
        "n_Treg_supported",
        "n_Treg_exploratory",
    ]
    ax = (
        summary
        .set_index(
            "sample"
        )[
            count_columns
        ]
        .plot(
            kind="bar",
            figsize=(15, 8),
            width=0.82,
        )
    )
    ax.set_title(
        "Strict, primary, and exploratory T/Treg counts"
    )
    ax.set_xlabel(
        "Sample"
    )
    ax.set_ylabel(
        "Cells"
    )
    ax.tick_params(
        axis="x",
        rotation=35,
    )
    ax.legend(
        fontsize=8,
        frameon=False,
    )
    plt.tight_layout()
    plt.savefig(
        OUTPUT_ROOT
        / "cross_sample_strict_primary_exploratory_counts.png",
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close()

    # Compare candidate Treg counts to the independent FOXP3-bin scale.
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(15, 6),
    )

    axes[0].scatter(
        summary[
            "FOXP3_positive_bins"
        ],
        summary[
            "n_Treg_supported"
        ],
        s=50,
    )
    for _, row in (
        summary.iterrows()
    ):
        axes[0].annotate(
            row[
                "sample"
            ],
            (
                row[
                    "FOXP3_positive_bins"
                ],
                row[
                    "n_Treg_supported"
                ],
            ),
            fontsize=7,
        )
    axes[0].set_xlabel(
        "FOXP3-positive 2-µm bins"
    )
    axes[0].set_ylabel(
        "Supported Treg candidates"
    )
    axes[0].set_title(
        "Supported Tregs versus independent FOXP3-bin signal"
    )

    axes[1].scatter(
        summary[
            "FOXP3_positive_bins"
        ],
        summary[
            "n_Treg_exploratory"
        ],
        s=50,
    )
    for _, row in (
        summary.iterrows()
    ):
        axes[1].annotate(
            row[
                "sample"
            ],
            (
                row[
                    "FOXP3_positive_bins"
                ],
                row[
                    "n_Treg_exploratory"
                ],
            ),
            fontsize=7,
        )
    axes[1].set_xlabel(
        "FOXP3-positive 2-µm bins"
    )
    axes[1].set_ylabel(
        "Exploratory Treg candidates"
    )
    axes[1].set_title(
        "Exploratory Tregs versus independent FOXP3-bin signal"
    )

    fig.tight_layout()
    fig.savefig(
        OUTPUT_ROOT
        / "Treg_candidates_vs_FOXP3_bin_reference.png",
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close()

    display(
        summary
    )
    display(
        paired
    )

manifest = {
    "pipeline_version": (
        PIPELINE_VERSION
    ),
    "method": (
        "strict-preserving inclusive T/Treg rescue"
    ),
    "n_samples_completed": int(
        len(
            results
        )
    ),
    "n_samples_failed": int(
        len(
            failures
        )
    ),
    "labels": {
        "strict_T": "fallback_T_lineage",
        "primary_T": "rescue_T_primary",
        "exploratory_T": "rescue_T_exploratory",
        "Treg_high_confidence": (
            "rescue_Treg_high_confidence"
        ),
        "Treg_supported": (
            "rescue_Treg_supported"
        ),
        "Treg_exploratory": (
            "rescue_Treg_exploratory"
        ),
    },
    "CD8B_used": False,
    "bin_reference_used_for_thresholds": False,
    "summary": str(
        OUTPUT_ROOT
        / "inclusive_rescue_summary_by_sample.csv"
    ),
    "paired_summary": str(
        OUTPUT_ROOT
        / "paired_Screen_C2D15_inclusive_rescue_summary.csv"
    ),
    "failures": failures,
}
write_json(
    manifest,
    OUTPUT_ROOT
    / "inclusive_rescue_manifest.json",
)

,sample,patient,cancer_type,biopsy_stage,n_cells,n_strict_T,n_primary_T,n_exploratory_T,n_raw_T_coherent,n_Treg_high_confidence,...,n_raw_FOXP3_in_exploratory_T,FOXP3_positive_bins,FOXP3_cells_bins_div3,CD8A_positive_bins,CD8A_cells_bins_div3,primary_T_spatially_mixed,supported_Treg_fraction_of_primary_T,exploratory_Treg_fraction_of_exploratory_T,supported_Treg_per_FOXP3_bin_scale,exploratory_Treg_per_FOXP3_bin_scale
0,Screen_39_21,patient_39_21,NSCLC,Screen,22950,1114,2039,2687,2490,7,...,110,191,64,444,148,0,0.045120,0.043543,1.437500,1.828125
1,C2D15_39_21,patient_39_21,NSCLC,C2D15,10822,1079,1172,1339,971,1,...,35,111,37,103,34,1,0.018771,0.018671,0.594595,0.675676
2,Screen_17_26,patient_17_26,NSCLC,Screen,47896,0,754,1350,1019,1,...,88,170,56,303,101,0,0.041114,0.042222,0.553571,1.017857
3,C2D15_17_26,patient_17_26,NSCLC,C2D15,87913,0,891,1627,1364,3,...,95,187,62,337,112,0,0.048260,0.040565,0.693548,1.064516
4,Screen_18_23,patient_18_23,melanoma,Screen,72386,147,923,1343,787,0,...,2,33,11,80,26,0,0.000000,0.000000,0.000000,0.000000
5,C2D15_18_23,patient_18_23,melanoma,C2D15,18594,34,821,1268,1063,0,...,3,79,26,327,109,0,0.007308,0.004732,0.230769,0.230769
6,Screen_16_22,patient_16_22,melanoma,Screen,70438,76,1244,2369,1099,0,...,49,80,26,267,89,1,0.001608,0.001688,0.076923,0.153846
7,C2D15_16_22,patient_16_22,melanoma,C2D15,76633,171,2554,4214,2734,9,...,186,327,109,623,207,1,0.050117,0.039630,1.174312,1.532110
8,Screen_30_16,patient_30_16,melanoma,Screen,66977,1690,4148,6096,4528,19,...,294,444,148,1615,538,1,0.049421,0.038550,1.385135,1.587838
9,C2D15_30_16,patient_30_16,melanoma,C2D15,351799,0,437,1241,412,0,...,27,103,34,393,131,0,0.045767,0.034649,0.588235,1.264706


,patient,cancer_type,Screen_n_strict_T,C2D15_n_strict_T,delta_C2D15_minus_Screen_n_strict_T,Screen_n_primary_T,C2D15_n_primary_T,delta_C2D15_minus_Screen_n_primary_T,Screen_n_exploratory_T,C2D15_n_exploratory_T,...,delta_C2D15_minus_Screen_n_Treg_exploratory,Screen_n_raw_FOXP3_positive,C2D15_n_raw_FOXP3_positive,delta_C2D15_minus_Screen_n_raw_FOXP3_positive,Screen_FOXP3_positive_bins,C2D15_FOXP3_positive_bins,delta_C2D15_minus_Screen_FOXP3_positive_bins,Screen_FOXP3_cells_bins_div3,C2D15_FOXP3_cells_bins_div3,delta_C2D15_minus_Screen_FOXP3_cells_bins_div3
0,patient_16_22,melanoma,76.0,171.0,95.0,1244.0,2554.0,1310.0,2369.0,4214.0,...,163.0,69.0,300.0,231.0,80.0,327.0,247.0,26.0,109.0,83.0
1,patient_17_26,NSCLC,0.0,0.0,0.0,754.0,891.0,137.0,1350.0,1627.0,...,9.0,149.0,167.0,18.0,170.0,187.0,17.0,56.0,62.0,6.0
2,patient_18_23,melanoma,147.0,34.0,-113.0,923.0,821.0,-102.0,1343.0,1268.0,...,6.0,25.0,68.0,43.0,33.0,79.0,46.0,11.0,26.0,15.0
3,patient_23_25,colon_cancer,4.0,109.0,105.0,99.0,115.0,16.0,231.0,126.0,...,-6.0,16.0,3.0,-13.0,22.0,5.0,-17.0,7.0,1.0,-6.0
4,patient_30_16,melanoma,1690.0,0.0,-1690.0,4148.0,437.0,-3711.0,6096.0,1241.0,...,-192.0,406.0,58.0,-348.0,444.0,103.0,-341.0,148.0,34.0,-114.0
5,patient_39_21,NSCLC,1114.0,1079.0,-35.0,2039.0,1172.0,-867.0,2687.0,1339.0,...,-92.0,165.0,80.0,-85.0,191.0,111.0,-80.0,64.0,37.0,-27.0


# How to use the result

## Recommended hierarchy for reporting

### Main sensitivity-oriented T-cell analysis

Use:

```python
rescue_T_primary
```

and keep:

```python
rescue_T_mixing_status
```

as a covariate/audit field. Consider excluding
`T_supported_spatially_mixed` cells from the most conservative DEG analysis,
then include them in sensitivity analysis.

### Treg analysis

Use three nested definitions:

```text
high-confidence:
    rescue_Treg_high_confidence

primary exploratory:
    rescue_Treg_supported

maximum sensitivity:
    rescue_Treg_exploratory
```

For any Screen-versus-C2D15 comparison, report at least the supported and
high-confidence definitions. The exploratory tier should be described as a
sensitivity analysis.

## Interpreting the bin reference

The `FOXP3_cells_bins_div3` column is a rough scale estimate, not a target and
not a true maximum. A genuine FOXP3-positive cell can contribute only one
positive 2-µm bin.

Useful questions are:

```text
Does candidate abundance broadly track FOXP3-bin abundance?
Are paired directions similar?
Does a threshold change add raw-FOXP3-supported candidates?
Are rescued candidates spatially plausible?
```

Do not tune each sample until its candidate count equals the bin estimate.

## DEG strategy

For a first-pass paired DEG:

```text
1. all primary rescued T cells
2. CD4 and CD8 subsets only where both stages have enough cells
3. supported Tregs as exploratory
4. high-confidence Tregs as validation/sensitivity
```

Patient/sample remains the biological replicate. Do not treat individual cells
as independent replicates.

## Stopping rule

This notebook deliberately produces nested evidence tiers rather than one
supposedly definitive answer. If the supported and exploratory tiers remain
concentrated in one sample or do not track raw/bin FOXP3 evidence, the data do
not support a reliable cross-sample Treg comparison.